# Workshop Data Prep — Vertem | 00 · Setup dos dados

Gera tabelas **RAW** mockadas para o domínio de **fidelidade e resgate de pontos** da Vertem, com problemas de qualidade propositais e campos de texto livre para demonstrar **AI Functions** e **Guardrails**. Inclui uma fonte **CDC** para o lab de AUTO CDC.

Rode uma **única vez** antes do workshop usando warehouse serverless ou cluster com Unity Catalog.

| Tabela | Linhas | Problemas embutidos | Uso em AI/Guardrails |
|---|---:|---|---|
| `clientes_b2b` | \~44 | casing/espaços no nome, e-mail nulo, datas em 2 formatos, tipo de programa e status inconsistentes, duplicatas | `descricao_empresa`: texto livre para `ai_classify` e `ai_summarize` |
| `parceiros_resgate` | 12 | nome/categoria/canal inconsistentes, SLA como texto, nulos | — |
| `campanhas` | \~126 | datas em 2 formatos, status/canal inconsistentes, orçamento como texto, duplicatas | `briefing_campanha`: texto livre para `ai_summarize` |
| `resgates` | \~25.208 | FKs órfãs, pontos/quantidade inválidos, datas em 2 formatos, status/canal inconsistentes, duplicatas | `feedback_participante`: texto com PII (CPF, telefone, e-mail) para **Guardrails** + `ai_classify` sentimento |

In [0]:
USE CATALOG vertem;
USE SCHEMA workshop_dataprep;

In [0]:
DROP TABLE IF EXISTS clientes_b2b; 
DROP TABLE IF EXISTS parceiros_resgate; 
DROP TABLE IF EXISTS campanhas; 
DROP TABLE IF EXISTS resgates; 

In [0]:
-- 1. Clientes B2B (clientes_b2b)
CREATE OR REPLACE TABLE clientes_b2b AS
WITH base AS (SELECT id AS n FROM range(1, 41)),
gen AS (
  SELECT
    n AS id_cliente,
    CASE pmod(n,4)
      WHEN 0 THEN concat('  ', element_at(array('BASF','NATURA','AMBEV','BRF','VIVO','CIELO','MAGALU','RAIA DROGASIL','LOCALIZA','STONE','PETROBRAS','XP INVESTIMENTOS'), cast(pmod(n,12)+1 as int)), '  ')
      WHEN 1 THEN lower(element_at(array('BASF','NATURA','AMBEV','BRF','VIVO','CIELO','MAGALU','RAIA DROGASIL','LOCALIZA','STONE','PETROBRAS','XP INVESTIMENTOS'), cast(pmod(n,12)+1 as int)))
      WHEN 2 THEN upper(element_at(array('BASF','NATURA','AMBEV','BRF','VIVO','CIELO','MAGALU','RAIA DROGASIL','LOCALIZA','STONE','PETROBRAS','XP INVESTIMENTOS'), cast(pmod(n,12)+1 as int)))
      ELSE initcap(lower(element_at(array('BASF','NATURA','AMBEV','BRF','VIVO','CIELO','MAGALU','RAIA DROGASIL','LOCALIZA','STONE','PETROBRAS','XP INVESTIMENTOS'), cast(pmod(n,12)+1 as int))))
    END AS nome_cliente,
    element_at(array('agro','Químicos','Bebidas','Foods','Telecom','Pagamentos','Varejo','Saúde','Mobilidade','Fintech','Energia','Serviços Financeiros'), cast(pmod(n,12)+1 as int)) AS setor,
    CASE WHEN pmod(n,7)=0 THEN NULL WHEN pmod(n,5)=0 THEN concat(' contato+', n, '@cliente.com ') ELSE concat('contato', n, '@cliente.com') END AS email_contato,
    CASE WHEN pmod(n,3)=0 THEN concat('(11) 3', lpad(cast(pmod(n*17,10000) as string),4,'0'), '-', lpad(cast(pmod(n*13,10000) as string),4,'0'))
         WHEN pmod(n,3)=1 THEN concat('11 3', lpad(cast(pmod(n*17,10000) as string),4,'0'), lpad(cast(pmod(n*13,10000) as string),4,'0'))
         ELSE concat('+55 11 3', lpad(cast(pmod(n*17,10000) as string),4,'0'), lpad(cast(pmod(n*13,10000) as string),4,'0')) END AS telefone_contato,
    element_at(array('São Paulo','Campinas','Rio de Janeiro','Curitiba','Porto Alegre','Belo Horizonte'), cast(pmod(n,6)+1 as int)) AS cidade_sede,
    CASE WHEN pmod(n,9)=0 THEN NULL ELSE element_at(array('SP','SP','RJ','PR','RS','MG'), cast(pmod(n,6)+1 as int)) END AS estado_sede,
    CASE WHEN pmod(n,2)=0 THEN date_format(date_add(date'2022-01-01', cast(pmod(n*23,700) as int)), 'dd/MM/yyyy') ELSE cast(date_add(date'2022-01-01', cast(pmod(n*23,700) as int)) AS string) END AS data_inicio_contrato,
    CASE pmod(n,6) WHEN 0 THEN 'Programa de Pontos' WHEN 1 THEN 'programa de pontos' WHEN 2 THEN 'Campanha Incentivo' WHEN 3 THEN 'campanha incentivo ' WHEN 4 THEN 'Cashback' ELSE NULL END AS tipo_programa,
    CASE pmod(n,5) WHEN 0 THEN 'Ativo' WHEN 1 THEN 'ATIVO' WHEN 2 THEN 'Piloto' WHEN 3 THEN 'expansão' ELSE NULL END AS status_cliente,
    element_at(array(
      'Multinacional química líder em insumos agrícolas. Atua em 90 países com foco em sustentabilidade. Contrato para equipe comercial com 1.200 colaboradores.',
      'Maior empresa de cosméticos do Brasil. Venda direta com revendedoras. Programa de incentivo para top consultoras com prêmios em viagens.',
      'Líder no setor de bebidas. Marcas Skol, Brahma e Guaraná. Campanha de pontos para distribuidores que atingem metas trimestrais.',
      'Gigante alimentício dona de Sadia e Perdigão. Programa de fidelidade B2B para redes de supermercados parceiras.',
      'Principal operadora de telecom do país. Pontos para clientes corporativos com mais de 50 linhas móveis.',
      'Empresa de meios de pagamento. Cashback para lojistas que processam acima de R$ 100 mil/mês.',
      'Varejista digital e físico com mais de 1.300 lojas. Parceiro de resgate e cliente B2B com campanha para vendedores.',
      'Maior rede de farmácias do Brasil. Incentivo para farmacêuticos que recomendam produtos da linha própria.',
      'Líder em locação de veículos. Pontos para clientes PJ que alugam frotas acima de 20 veículos.',
      'Fintech de pagamentos. Recompensas para comerciantes que adotam a maquininha como meio exclusivo.',
      'Estatal de energia. Parceira de resgate no abastecimento e cliente B2B com campanha para postos bandeirados.',
      'Corretora e banco de investimentos. Cashback em pontos para assessores que captam novos clientes.'
    ), cast(pmod(n,12)+1 as int)) AS descricao_empresa
  FROM base
),
dups AS (SELECT * FROM gen WHERE pmod(id_cliente, 10) = 0)
SELECT * FROM gen UNION ALL SELECT * FROM dups;

In [0]:
-- 2a. Parceiros de resgate (parceiros_resgate)
CREATE OR REPLACE TABLE parceiros_resgate AS
SELECT
  n AS id_parceiro,
  CASE pmod(n,4)
    WHEN 0 THEN concat('  ', element_at(array('Magazine Luiza','Shell','Petrobras','iFood','Uber','Decathlon','Netshoes','Azul','LATAM Pass','Carrefour','C&A','Petz'), cast(pmod(n,12)+1 as int)), '  ')
    WHEN 1 THEN lower(element_at(array('Magazine Luiza','Shell','Petrobras','iFood','Uber','Decathlon','Netshoes','Azul','LATAM Pass','Carrefour','C&A','Petz'), cast(pmod(n,12)+1 as int)))
    WHEN 2 THEN upper(element_at(array('Magazine Luiza','Shell','Petrobras','iFood','Uber','Decathlon','Netshoes','Azul','LATAM Pass','Carrefour','C&A','Petz'), cast(pmod(n,12)+1 as int)))
    ELSE element_at(array('Magazine Luiza','Shell','Petrobras','iFood','Uber','Decathlon','Netshoes','Azul','LATAM Pass','Carrefour','C&A','Petz'), cast(pmod(n,12)+1 as int))
  END AS nome_parceiro,
  CASE pmod(n,6)
    WHEN 0 THEN 'Varejo' WHEN 1 THEN 'Combustível' WHEN 2 THEN 'combustivel '
    WHEN 3 THEN 'delivery' WHEN 4 THEN 'Viagem' ELSE NULL END AS categoria_parceiro,
  CASE pmod(n,4) WHEN 0 THEN 'API' WHEN 1 THEN 'api' WHEN 2 THEN 'Batch' ELSE 'Marketplace' END AS canal_liquidacao,
  CASE pmod(n,5) WHEN 0 THEN '24h' WHEN 1 THEN '48h' WHEN 2 THEN '72 horas' WHEN 3 THEN NULL ELSE 'D+1' END AS sla_resgate,
  CASE pmod(n,4) WHEN 0 THEN 'Ativo' WHEN 1 THEN 'ATIVO' WHEN 2 THEN 'Homologação' ELSE NULL END AS status_parceiro
FROM (SELECT id AS n FROM range(1, 13));

-- 2b. Catálogo de recompensas (catalogo_recompensas)
CREATE OR REPLACE TABLE catalogo_recompensas AS
SELECT
  n AS id_recompensa,
  CASE WHEN pmod(n,17)=0 THEN 9000 + n ELSE cast(pmod(n,12)+1 as bigint) END AS id_parceiro,
  CASE pmod(n,8)
    WHEN 0 THEN concat('Vale Compra R$ ', cast((pmod(n*13,8)+1)*50 as string))
    WHEN 1 THEN concat('Abastecimento ', cast((pmod(n*7,5)+1)*20 as string), ' litros')
    WHEN 2 THEN concat('Cashback ', cast((pmod(n*9,6)+1)*25 as string))
    WHEN 3 THEN concat('Milhas ', cast((pmod(n*11,8)+1)*1000 as string))
    WHEN 4 THEN concat('Gift Card ', element_at(array('Magalu','Shell','iFood','Uber','Petz'), cast(pmod(n,5)+1 as int)))
    WHEN 5 THEN concat('Fone Bluetooth ', lpad(cast(n as string),3,'0'))
    WHEN 6 THEN concat('Cafeteira ', lpad(cast(n as string),3,'0'))
    ELSE concat('Kit Experiência ', lpad(cast(n as string),3,'0'))
  END AS nome_recompensa,
  CASE pmod(n,7)
    WHEN 0 THEN 'Gift Card' WHEN 1 THEN 'gift card' WHEN 2 THEN 'Combustível'
    WHEN 3 THEN 'combustivel ' WHEN 4 THEN 'Milhas' WHEN 5 THEN 'Produto Físico' ELSE NULL END AS categoria_recompensa,
  CASE pmod(n,4)
    WHEN 0 THEN concat('R$ ', format_number((pmod(n*97,40000) + 5000) / 1.0, 0))
    WHEN 1 THEN cast(pmod(n*97,40000) + 5000 as string)
    WHEN 2 THEN NULL
    ELSE concat(cast(pmod(n*97,40000) + 5000 as string), ' pontos') END AS custo_pontos,
  CASE pmod(n,5)
    WHEN 0 THEN concat('R$ ', format_number((pmod(n*31,700) + 29.9), 2))
    WHEN 1 THEN cast(round(pmod(n*31,700) + 29.9, 2) as string)
    WHEN 2 THEN NULL
    ELSE concat('R$', cast(round(pmod(n*31,700) + 29.9, 2) as string)) END AS preco_referencia_reais,
  CASE pmod(n,4) WHEN 0 THEN 'Ativo' WHEN 1 THEN 'ATIVO' WHEN 2 THEN 'Em revisão' ELSE NULL END AS status_item,
  -- Campo texto livre para demo de AI (ai_extract características)
  element_at(array(
    'Vale compra digital válido por 60 dias em todas as lojas físicas e online do parceiro. Não cumulativo com outras promoções.',
    'Crédito para abastecimento em postos credenciados da rede. Válido para gasolina, etanol e diesel S-10.',
    'Cashback depositado em até 48h na conta digital do participante. Disponível para transferência via PIX.',
    'Milhas creditadas em até 15 dias úteis no programa do parceiro aéreo. Podem ser usadas em voos nacionais e internacionais.',
    'Gift card digital com código único. Envio por e-mail em até 2h após o resgate. Válido por 12 meses.',
    'Fone de ouvido bluetooth com cancelamento de ruído ativo. Bateria de 30h. Entrega em até 10 dias úteis. Garantia de 1 ano.',
    'Cafeteira elétrica com moedor integrado. 15 bar de pressão. Compatível com cápsulas e pó. Entrega express em capitais.',
    'Kit experiência gastronomia inclui jantar para 2 em restaurante parceiro. Reserva online obrigatória com 48h de antecedência.'
  ), cast(pmod(n,8)+1 as int)) AS descricao_item
FROM (SELECT id AS n FROM range(1, 181));

In [0]:
-- 3. Campanhas (campanhas)
CREATE OR REPLACE TABLE campanhas AS
WITH base AS (SELECT id AS n FROM range(1, 121)),
gen AS (
  SELECT
    n AS id_campanha,
    CASE WHEN pmod(n,22)=0 THEN 8000 + n ELSE cast(pmod(n*5,40)+1 as bigint) END AS id_cliente,
    concat(
      element_at(array('Bônus de Boas-Vindas','Resgate Turbo','Aniversário do Cliente','Meta Batida','Campanha Regional','Pontos em Dobro','Volta às Aulas','Black Friday Rewards'), cast(pmod(n,8)+1 as int)),
      ' ',
      element_at(array('Q1','Q2','Q3','Q4'), cast(pmod(n,4)+1 as int))
    ) AS nome_campanha,
    CASE pmod(n,6)
      WHEN 0 THEN 'acúmulo' WHEN 1 THEN 'Acumulo' WHEN 2 THEN 'resgate'
      WHEN 3 THEN 'Resgate ' WHEN 4 THEN 'engajamento' ELSE NULL END AS tipo_campanha,
    CASE WHEN pmod(n,2)=0 THEN date_format(date_add(date'2024-01-01', cast(pmod(n*11,420) as int)), 'dd/MM/yyyy') ELSE cast(date_add(date'2024-01-01', cast(pmod(n*11,420) as int)) as string) END AS data_inicio,
    CASE WHEN pmod(n,2)=0 THEN date_format(date_add(date'2024-02-15', cast(pmod(n*11,420) as int)), 'dd/MM/yyyy') ELSE cast(date_add(date'2024-02-15', cast(pmod(n*11,420) as int)) as string) END AS data_fim,
    CASE pmod(n,5)
      WHEN 0 THEN concat('R$ ', format_number((pmod(n*111,900000) + 50000) / 1.0, 0))
      WHEN 1 THEN cast(pmod(n*111,900000) + 50000 as string)
      WHEN 2 THEN NULL
      WHEN 3 THEN concat(cast(pmod(n*111,900000) + 50000 as string), ',00')
      ELSE concat('R$', cast(pmod(n*111,900000) + 50000 as string)) END AS orcamento_incentivo,
    CASE pmod(n,6) WHEN 0 THEN 'App' WHEN 1 THEN 'app' WHEN 2 THEN 'Portal' WHEN 3 THEN 'WhatsApp' WHEN 4 THEN 'whatsapp' ELSE NULL END AS canal_principal,
    CASE pmod(n,5) WHEN 0 THEN 'Ativa' WHEN 1 THEN 'ATIVA' WHEN 2 THEN 'Pausada' WHEN 3 THEN 'Encerrada' ELSE NULL END AS status_campanha,
    -- Campo texto livre para demo de AI (ai_summarize briefing)
    element_at(array(
      'Campanha de boas-vindas para novos participantes do programa de fidelidade. Cada cadastro recebe 500 pontos de bônus instantâneo. Objetivo: aumentar base em 20% no trimestre. Público-alvo: colaboradores recém-contratados dos clientes B2B.',
      'Promoção relampâgo de 48h com pontos em dobro em todos os resgates. Meta: queimar 2 milhões de pontos represados no saldo dos participantes. Divulgação via push e WhatsApp.',
      'Campanha exclusiva para aniversários do mês. O participante ganha 1.000 pontos extras ao resgatar qualquer item do catálogo no mês do aniversário. Ativação automática via data de nascimento.',
      'Incentivo para equipes que atingem metas trimestrais. Cada meta batida libera um resgate exclusivo de maior valor. Reportar resultados ao gerente de conta em até 5 dias úteis.',
      'Campanha geo-segmentada para regiões Sul e Sudeste. Parceiros locais oferecem condições especiais. Budget limitado a R$ 150 mil. Validade de 30 dias corridos.',
      'Mecnica simples: a cada resgate, o participante recebe o dobro dos pontos gastos de volta em 30 dias. Teto de 10.000 pontos por CPF. Orçamento mensal de R$ 200 mil. Comunicar via app e e-mail.',
      'Volta às aulas com foco em gift cards de livrarias e artigos de papelaria. Parceiros: Saraiva e Kalunga. Desconto de 15% em pontos para esses itens durante fevereiro.',
      'Black Friday do programa: 70% de desconto em pontos nos 20 itens mais resgatados. Estoque limitado. Campanha válida apenas no dia 29/11. Previsão de 50 mil transações.'
    ), cast(pmod(n,8)+1 as int)) AS briefing_campanha
  FROM base
),
dups AS (SELECT * FROM gen WHERE pmod(id_campanha, 20) = 0)
SELECT * FROM gen UNION ALL SELECT * FROM dups;

In [0]:
-- 4. Resgates (resgates)
CREATE OR REPLACE TABLE resgates AS
WITH gen AS (
  SELECT
    n AS id_resgate,
    CASE WHEN pmod(n,21)=0 THEN 9000 + n ELSE cast(pmod(n*5,40)+1 as bigint) END AS id_cliente,
    CASE WHEN pmod(n,27)=0 THEN 7000 + n ELSE cast(pmod(n*7,120)+1 as bigint) END AS id_campanha,
    CASE WHEN pmod(n,29)=0 THEN 5000 + n ELSE cast(pmod(n*11,12)+1 as bigint) END AS id_parceiro,
    CASE WHEN pmod(n,31)=0 THEN 8000 + n ELSE cast(pmod(n*13,180)+1 as bigint) END AS id_recompensa,
    cast(100000 + pmod(n*17, 8000) as bigint) AS id_participante,
    CASE WHEN pmod(n,41)=0 THEN -1 * (pmod(n*97, 40000) + 5000) ELSE pmod(n*97, 40000) + 5000 END AS pontos_resgatados,
    CASE WHEN pmod(n,37)=0 THEN 0 WHEN pmod(n,43)=0 THEN -1 ELSE cast(pmod(n,4)+1 as int) END AS quantidade,
    CASE WHEN pmod(n,2)=0 THEN date_format(date_add(date'2025-01-01', cast(pmod(n,240) as int)), 'dd/MM/yyyy') ELSE cast(date_add(date'2025-01-01', cast(pmod(n,240) as int)) as string) END AS data_resgate,
    CASE pmod(n,6) WHEN 0 THEN 'Solicitado' WHEN 1 THEN 'SOLICITADO' WHEN 2 THEN 'Aprovado' WHEN 3 THEN 'APROVADO' WHEN 4 THEN 'Cancelado' ELSE NULL END AS status_resgate,
    CASE pmod(n,6) WHEN 0 THEN 'App' WHEN 1 THEN 'app' WHEN 2 THEN 'Portal' WHEN 3 THEN 'WhatsApp' WHEN 4 THEN 'Parceiro' ELSE NULL END AS canal_resgate,
    CASE WHEN pmod(n,8)=0 THEN 'Black Friday' WHEN pmod(n,11)=0 THEN 'Aniversário' ELSE NULL END AS tag_campanha,
    -- Campo texto livre com PII proposital para demo de GUARDRAILS + ai_classify sentimento
    element_at(array(
      'Adorei a experiência de resgate! Super rápido e fácil. Meu CPF é 123.456.789-00 caso precisem para o cadastro.',
      'Produto chegou com defeito, preciso de suporte urgente. Me liguem no (11) 98765-4321 por favor.',
      'Excelente! Melhor programa de pontos que já participei. Recomendo a todos os colegas.',
      'Demora absurda na entrega. Já se passaram 15 dias e nada do vale-compra. Péssimo serviço.',
      'Resgate realizado sem problemas. Parabéns pela agilidade da equipe!',
      'Não recebi o código do gift card. Enviem para maria.santos@gmail.com por favor.',
      'Péssimo atendimento no parceiro. Fui mal tratado na loja da Magazine Luiza do Shopping Morumbi.',
      'Muito bom! Consegui resgatar milhas da Azul para viajar com a família no fim do ano.',
      'O saldo de pontos não bateu. Tinha 45.000 e só apareceram 32.000. Meu RG é 12.345.678-9 e CPF 321.654.987-00.',
      'Simplesmente perfeito. Já indiquei para todos os colegas da empresa. Nota 10!',
      NULL,
      'Gostaria de mais opções de parceiros na região Norte. Aqui só tem Shell e Carrefour.',
      'O app travou durante o resgate e perdi os pontos. URGENTE! Minha conta bancária é ag 0001 cc 12345-6 Banco do Brasil.',
      'Achei o catálogo limitado, mas o cashback na Shell valeu a pena. Quero mais promoções assim.',
      'Tudo certo com o resgate. Nota 10! Entrega rápida e produto conforme descrição.',
      'Cancelei porque encontrei mais barato no site direto. Devolvam meus 8.500 pontos imediatamente.',
      NULL,
      'Atendente muito prestativo, resolveu tudo em 5 minutos. Obrigado Carlos da equipe de suporte!',
      'Quero registrar reclamação formal. Meu nome é João Pedro Silva, CPF 987.654.321-00, telefone (21) 99876-5432.',
      'Amei o vale-compra do Magalu! Já quero resgatar outro. Podem mandar o próximo para joao.pedro@empresa.com.br.'
    ), cast(pmod(n, 20) + 1 as int)) AS feedback_participante
  FROM (SELECT id AS n FROM range(1, 25001))
),
dups AS (SELECT * FROM gen WHERE pmod(id_resgate, 120) = 0)
SELECT * FROM gen UNION ALL SELECT * FROM dups;

In [0]:
-- 6. Conferência final
SELECT 'clientes_b2b' AS tabela, count(*) AS linhas FROM clientes_b2b
UNION ALL SELECT 'parceiros_resgate', count(*) FROM parceiros_resgate
UNION ALL SELECT 'campanhas', count(*) FROM campanhas
UNION ALL SELECT 'resgates', count(*) FROM resgates
ORDER BY tabela;